# 🚀 Slide 1 — What is SimpliSQL?

## The Big Idea
**SimpliSQL** is a 100% offline, AI-powered desktop SQL assistant built for analysts, accountants, and data teams who work with messy files — Excel, CSV, JSON, XML — but don't want to touch cloud services or pay per query.

---

## Core Value Proposition
| Problem | SimpliSQL Solution |
|---|---|
| Analysts drown in Excel files | Upload any file → instant SQL table |
| Writing SQL is hard | Ask in plain English → AI writes the query |
| Cloud tools leak sensitive data | Runs 100% locally — no internet required |
| Switching between tools wastes time | Chat + Query + Export in one window |

---

## Tech Stack at a Glance
- **UI**: PyQt6 desktop app (Windows)
- **Database**: DuckDB (blazing-fast in-process SQL)
- **AI**: llama-cpp-python running GGUF models (Gemma, TinyLlama, SQLCoder)
- **Storage**: Apache Parquet via pyarrow
- **File Ingestion**: pandas (Excel/CSV/JSON/XML/ZIP → Parquet)

# 🏗️ Slide 2 — Architecture & How It Works

## System Flow

```
User uploads file
    │
    ▼
pandas reads CSV / Excel / JSON / XML / ZIP
    │
    ▼
DuckDB registers DataFrame → writes to .parquet
    │
    ▼
User types a question in plain English
    │
    ▼
AI Assistant (local GGUF model via llama-cpp-python)
    │  reads table schema + last 6 chat turns
    │  generates DuckDB-compatible SQL
    ▼
DuckDB executes query → results shown in grid
    │
    ▼
User exports → CSV / Excel / PDF / Chart
```

---

## Key Architectural Decisions

1. **Parquet as internal format** — DuckDB queries parquet 10–100× faster than raw CSV
2. **Streaming token output** — AI response appears word-by-word, no frozen UI
3. **Schema-aware prompt** — system prompt injects actual column names + types so the AI generates correct SQL
4. **Local models only** — no API keys, no data leaves the machine
5. **Worker threads** — all AI inference runs in `QThread` to keep UI responsive

# ⚡ Slide 3 — Build It in a Hackathon (Step-by-Step)

## Prerequisites
```bash
Python 3.11+
pip install duckdb pandas pyarrow PyQt6 llama-cpp-python huggingface-hub openpyxl pyperclip reportlab matplotlib plotly
```

---

## Phase 1 — File Ingestion (Hour 1–2)
```python
import pandas as pd
import duckdb

# Read any file
df = pd.read_excel('data.xlsx', sheet_name='Sheet1')

# Register with DuckDB and save as Parquet
conn = duckdb.connect()
conn.register('temp_table', df)
conn.execute("COPY temp_table TO 'data.parquet' (FORMAT 'parquet', COMPRESSION 'SNAPPY');")
```

---

## Phase 2 — SQL Execution Engine (Hour 2–3)
```python
# Query the parquet file directly
result = conn.execute("SELECT * FROM 'data.parquet' LIMIT 10").fetchdf()
print(result)
```

---

## Phase 3 — Local AI (Hour 3–5)
```python
from llama_cpp import Llama

# Load a small GGUF model (e.g. TinyLlama 1.1B ~700MB)
llm = Llama(model_path='models/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf',
            n_ctx=2048, n_threads=4)

schema = "Table: sales | Columns: date TEXT, product TEXT, amount FLOAT"
question = "Show me total sales by product"

prompt = f"You are a DuckDB SQL expert.\nSchema: {schema}\nQuestion: {question}\nSQL:"

output = llm(prompt, max_tokens=200, stop=[";"])
sql = output['choices'][0]['text'].strip()
print(sql)
```

---

## Phase 4 — PyQt6 UI (Hour 5–8)
- `QMainWindow` with a `QSplitter`: left = file tree, right = query editor + results grid
- `QThread` for AI inference (never block the main thread)
- `QTableWidget` or `QTableView` + `QAbstractTableModel` to display query results
- `QTextEdit` for streaming AI chat output

# 🤖 Slide 4 — AI Layer Deep Dive

## Choosing the Right GGUF Model for a Hackathon

| Model | Size | RAM Needed | Best For |
|---|---|---|---|
| TinyLlama 1.1B Q4 | ~700 MB | 2 GB | Fast demo, simple SQL |
| Gemma-2 2B Q4 | ~1.5 GB | 4 GB | Better reasoning, still fast |
| Phi-2 2.7B Q4 | ~1.8 GB | 4 GB | Code + SQL quality |
| SQLCoder 7B Q4 | ~4.5 GB | 8 GB | Best SQL accuracy |
| Gemma-4 4B Q4 | ~2.5 GB | 6 GB | Latest, best overall |

> **Hackathon tip**: Start with TinyLlama for speed during demos. Switch to Gemma-2 2B for quality.

---

## Prompt Engineering for SQL Generation

The secret is injecting the **actual schema** into every request:

```python
def build_system_prompt(table_schemas: dict) -> str:
    schema_text = ""
    for table_name, columns in table_schemas.items():
        col_list = ", ".join(f"{col} {dtype}" for col, dtype in columns)
        schema_text += f"Table: {table_name} | Columns: {col_list}\n"

    return f"""You are a DuckDB SQL expert assistant.
Available tables and their schemas:
{schema_text}
Rules:
- Always use DuckDB syntax (not MySQL/PostgreSQL)
- Use double quotes for column names with spaces
- Return only the SQL query, no explanation
- End every query with a semicolon"""
```

---

## Streaming Tokens with QThread

```python
from PyQt6.QtCore import QThread, pyqtSignal

class AIChatThread(QThread):
    token_ready = pyqtSignal(str)      # emit each token
    response_done = pyqtSignal(str)    # emit full response

    def run(self):
        for chunk in self.llm(self.prompt, stream=True, max_tokens=512):
            token = chunk['choices'][0]['text']
            self.token_ready.emit(token)       # UI appends token live
        self.response_done.emit(self.full_text)
```

# 🏆 Slide 5 — Hackathon Winning Tips & Demo Script

## What Judges Look For
1. **Live demo works** — nothing kills a pitch faster than a crash
2. **Real problem solved** — "analysts waste 2 hrs/day writing SQL"
3. **Technical depth** — local AI + DuckDB shows genuine engineering
4. **Privacy angle** — zero cloud, zero data leakage resonates strongly in 2025

---

## 3-Minute Demo Script

| Time | Action | What to Say |
|---|---|---|
| 0:00 | Show an Excel file with 10,000 rows | "This is a real GST reconciliation file from an accountant" |
| 0:30 | Drag & drop into SimpliSQL | "It uploads and converts to a query-ready table in seconds" |
| 1:00 | Type: *"Show top 5 vendors by total invoice value"* | "I just ask in plain English — no SQL needed" |
| 1:30 | AI streams the SQL + results appear | "The AI runs locally — this never left my laptop" |
| 2:00 | Click Export → Excel | "One click to export back to Excel for my client" |
| 2:30 | Open Settings → switch model | "I can swap AI models based on speed vs accuracy needs" |

---

## Minimum Viable Hackathon Build Checklist

- [ ] File upload: CSV + Excel → DuckDB
- [ ] SQL editor with run button
- [ ] Results grid (paginated)
- [ ] AI chat: plain English → SQL (TinyLlama)
- [ ] Export to CSV
- [ ] One working end-to-end demo dataset

---

## Differentiators to Highlight

- **No internet required** — works in secure enterprise environments
- **No subscription** — one-time install, bring your own model
- **Handles messy data** — auto-fixes mixed types, bad columns, encoding issues
- **DuckDB speed** — queries 1M rows in milliseconds on a laptop
- **Extensible** — swap any GGUF model, add new file formats

---

> 💡 **NotebookLM Tip**: Upload this notebook plus your `README.md` and `ARCHITECTURE.md` to NotebookLM. Ask it to generate a podcast-style audio overview, a FAQ, or a study guide for your pitch prep.